In [1]:
# Step 1: Import Libraries
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


In [4]:
# Step 2: Load Preprocessed Data
X = np.load("savedPackages/padded_sequences_withNegations.npy")
y = np.load("savedPackages/labels_withNegations.npy")

with open("savedPackages/tokenizer_withNegations.pkl", "rb") as f:
    tokenizer = pickle.load(f)


In [6]:
# Step 3: Load GloVe Embeddings
embedding_index = {}
with open("embeddings/glove.6B.100d.txt", encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embedding_index[word] = coefs

embedding_dim = 100
vocab_size = 20000
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in tokenizer.word_index.items():
    if i < vocab_size:
        embedding_vector = embedding_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector


In [7]:
# Step 4: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [8]:
# Step 5: Build ANN Model
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim,
                    weights=[embedding_matrix], input_length=X.shape[1], trainable=False))
model.add(Flatten())
model.add(Dense(256, activation='relu', kernel_regularizer=l2(0.01)))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(128, activation='relu', kernel_regularizer=l2(0.01)))
model.add(Dropout(0.5))
model.add(Dense(3, activation='softmax'))  # 3 classes: NEG, POS, NEU


D:\SentimentAnalyzer_Project\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [9]:
# Step 6: Compile and Train
model.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(X_train, y_train, validation_split=0.2, epochs=10, batch_size=128, callbacks=[early_stop])


Epoch 1/10
5900/5900 ━━━━━━━━━━━━━━━━━━━━ 239s 39ms/step - accuracy: 0.8053 - loss: 1.2140 - val_accuracy: 0.8376 - val_loss: 0.6179
Epoch 2/10
5900/5900 ━━━━━━━━━━━━━━━━━━━━ 233s 40ms/step - accuracy: 0.8277 - loss: 0.6329 - val_accuracy: 0.8415 - val_loss: 0.5624
Epoch 3/10
5900/5900 ━━━━━━━━━━━━━━━━━━━━ 220s 37ms/step - accuracy: 0.8286 - loss: 0.6026 - val_accuracy: 0.8522 - val_loss: 0.5357
Epoch 4/10
5900/5900 ━━━━━━━━━━━━━━━━━━━━ 213s 36ms/step - accuracy: 0.8297 - loss: 0.5898 - val_accuracy: 0.8409 - val_loss: 0.5513
Epoch 5/10
5900/5900 ━━━━━━━━━━━━━━━━━━━━ 253s 35ms/step - accuracy: 0.8311 - loss: 0.5835 - val_accuracy: 0.8453 - val_loss: 0.5304
Epoch 6/10
5900/5900 ━━━━━━━━━━━━━━━━━━━━ 178s 30ms/step - accuracy: 0.8286 - loss: 0.5798 - val_accuracy: 0.8516 - val_loss: 0.5205
Epoch 7/10
5900/5900 ━━━━━━━━━━━━━━━━━━━━ 176s 30ms/step - accuracy: 0.8307 - loss: 0.5745 - val_accuracy: 0.8478 - val_loss: 0.5197
Epoch 8/10
5900/5900 ━━━━━━━━━━━━━━━━━━━━ 223s 38ms/step - accuracy: 

In [10]:
# Step 7: Evaluate Model
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {acc:.4f}")

y_pred = model.predict(X_test)
y_pred_labels = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_labels, target_names=['NEG', 'POS', 'NEU']))


7375/7375 ━━━━━━━━━━━━━━━━━━━━ 42s 6ms/step - accuracy: 0.8531 - loss: 0.5081
Test Accuracy: 0.8530
7375/7375 ━━━━━━━━━━━━━━━━━━━━ 24s 3ms/step
              precision    recall  f1-score   support

         NEG       0.75      0.72      0.73     35875
         POS       0.84      0.83      0.84     74094
         NEU       0.89      0.91      0.90    126023

    accuracy                           0.85    235992
   macro avg       0.83      0.82      0.82    235992
weighted avg       0.85      0.85      0.85    235992



In [11]:
model.save("ann_model_with_negations.keras")


In [12]:
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model


In [13]:
# Load tokenizer
with open("savedPackages/tokenizer_withNegations.pkl", "rb") as f:
    tokenizer = pickle.load(f)

# Load model (if you saved it)
model = tf.keras.models.load_model("ann_model_with_negations.keras")


In [14]:
import re

def handle_negation(text):
    text = re.sub(r"n't", " not", text)
    text = re.sub(r'(?i)(not|no)\s+(\w+)', lambda m: m.group(1) + '_' + m.group(2), text)
    return text

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|@\S+|#\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9_ ]", "", text)
    text = handle_negation(text)
    return text.strip()


In [29]:
from tensorflow.keras.preprocessing.sequence import pad_sequences



custom_text = "I'm so excited about our trip to Paris"
cleaned = clean_text(custom_text)
sequence = tokenizer.texts_to_sequences([cleaned])
padded = pad_sequences(sequence, maxlen=50)



In [30]:
pred = model.predict(padded)
pred_class = np.argmax(pred, axis=1)[0]

label_map = {0: "NEGATIVE", 1: "POSITIVE", 2: "NEUTRAL"}
print(f"Predicted Sentiment: {label_map[pred_class]}")



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
Predicted Sentiment: NEUTRAL
